# Space Weather Launch Safety Predictor
## IBM Bob Generated Code — Audit Notebook

This notebook contains the full pipeline code generated by IBM Bob as an audit and learning artifact.
The production source code lives in `src/` and `dashboard/`.

---
> **Educational Disclaimer:** This is an educational risk-assessment system.
> The risk scores and recommendations are derived from a simplified educational model
> and are NOT suitable for real spacecraft launch decisions.

## Task 1 — Install and Verify Libraries

In [ ]:
import sys
import subprocess

required = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'streamlit', 'joblib', 'requests']

for pkg in required:
    try:
        mod = __import__(pkg.replace('-', '_'))
        version = getattr(mod, '__version__', 'installed')
        print(f'  {pkg:<20} {version}')
    except ImportError:
        print(f'  {pkg:<20} NOT FOUND — install with: pip install {pkg}')

## Task 2 — Download and Load Dataset

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.data_loader import load_dataset
df = load_dataset()

## Task 3 — Prepare Data

In [ ]:
from src.data_cleaning import clean_space_weather_data
space_df = clean_space_weather_data(df)
print(space_df.dtypes)
space_df.head(5)

## Task 4 — Explore and Analyze Data

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Change to 'inline' if running in Jupyter with display
import matplotlib.pyplot as plt
%matplotlib inline

from src.eda import run_eda
eda_stats = run_eda(space_df)

## Task 5 — Feature Engineering

In [ ]:
from src.feature_engineering import build_risk_features
risk_features_df = build_risk_features(space_df)
print(risk_features_df.shape)
risk_features_df.head(5)

## Task 6 — Risk Scoring

In [ ]:
from src.risk_scoring import apply_risk_scoring
scored_df = apply_risk_scoring(risk_features_df)
scored_df[['date', 'risk_score', 'risk_level', 'recommendation']].tail(10)

## Task 7 — Train and Evaluate Decision Model

In [ ]:
from src.model_training import train_model, FEATURE_COLS
from src.model_evaluation import evaluate_model

model, X_train, X_test, y_train, y_test, y_pred = train_model(scored_df)
results = evaluate_model(model, X_train, X_test, y_train, y_test, y_pred, FEATURE_COLS)

## Task 8 — Save Model and Risk Data

In [ ]:
from src.persistence import save_artifacts
save_artifacts(model, scored_df, FEATURE_COLS)

## Task 9 — Date-Range Go/No-Go Dashboard

The full interactive dashboard runs via Streamlit:

```bash
streamlit run dashboard/app.py
```

Below is an inline preview using the saved data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.prediction import load_saved_data
from dashboard.dashboard_utils import filter_date_range, compute_date_range_summary
from dashboard.charts import (
    chart_risk_score_per_day,
    chart_daily_recommendation,
    chart_solar_events_48h,
)

data = load_saved_data()
full_df = data['recent_features'].copy()
full_df['date'] = pd.to_datetime(full_df['date'])

# Use the full available date range
start = full_df['date'].min()
end   = full_df['date'].max()

filtered = filter_date_range(full_df, start, end)
summary  = compute_date_range_summary(filtered)

print('Date range:', start.date(), '→', end.date())
print('Total days:', summary['total_days'])
print('Avg risk score:', summary['avg_risk_score'])
print('GO:', summary['go_days'], 'CAUTION:', summary['caution_days'],
      'DELAY:', summary['delay_days'], 'NO-GO:', summary['no_go_days'])
print('Overall recommendation:', summary['overall_recommendation'])

fig1 = chart_risk_score_per_day(filtered)
plt.show()

fig2 = chart_daily_recommendation(filtered)
plt.show()

fig3 = chart_solar_events_48h(filtered)
plt.show()